# Neural Hydrology — 5-Condition Factorial Run on Colab

Single-click run of the **5-condition factorial ablation** on Component 0 (183 basins).

## Conditions (15 runs total = 5 conditions × 3 seeds)

| ID | Architecture | Topology features | Message passing | Variant flag |
|---|---|---|---|---|
| **L** | NH `cudalstm` | — | — | (NH config) |
| **G** | DirectedGraphLSTM, empty edges | — | — | `empty_graph` |
| **G+T** | DirectedGraphLSTM, empty edges | ✓ | — | `topology_features` |
| **G+M** | DirectedGraphLSTM, full edges | — | ✓ | `warm` |
| **G+T+M** | DirectedGraphLSTM, full edges | ✓ | ✓ | `full_graph_with_topology` |

Seeds: 11, 13, 17.

## Configuration

Two variables to adjust if needed (Cell 2):
- `DRIVE_CAMELS_PATH` — where your `camels_us/` folder lives on Drive (auto-detected if blank)
- `MODE` — `'demo'` (1 seed × 5 conditions) or `'full'` (3 seeds × 5 conditions)

## How to use

1. Runtime → Change runtime type → **T4 GPU** (cheapest; full sweep fits in ~30 hr / ~45 units). Save.
2. Runtime → Run all.
3. Wait for the final summary cell to print `summary.json`.

Notebook is **idempotent** — every condition × seed checks for completed outputs before retraining.

## Defenses against the bugs we hit in the prior A/B/C run

- All training writes directly to `runs/5cond_factorial/<cond>_seed<N>/` via `--run-dir`, so naming is fully deterministic and skip-if-done is exact (no glob over timestamp suffixes).
- Smoke-test cell deletes its `_SMOKE_` folders immediately after the smoke pass — cannot pollute later glob checks.
- Skip-if-done explicitly excludes `_SMOKE_` (defense in depth) even though the new run-dir scheme already keeps smoke folders out of the production paths.
- Condition L runs `train` then `evaluate` so `test_metrics.csv` exists for the analysis cell.
- All graph variants train with `--use-compile` for the torch.compile forward-pass speedup (Phase 1.2).


## Cell 1 — Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Cell 2 — Configuration

In [2]:
import os

# === USER CONFIG ===
GITHUB_URL = 'https://github.com/Op-2005/neural_hydro.git'
DRIVE_CAMELS_PATH = ''  # leave empty for auto-detection
MODE = 'full'           # 'demo' (1 seed) or 'full' (3 seeds)
# ====================

AUTO_DETECT_CANDIDATES = [
    '/content/drive/MyDrive/datasets/camels_us',
    '/content/drive/MyDrive/neural_hydro/datasets/camels_us',
    '/content/drive/MyDrive/neural_hydrology/datasets/camels_us',
    '/content/drive/MyDrive/camels_us',
    '/content/drive/MyDrive/data/camels_us',
]
if not DRIVE_CAMELS_PATH:
    for cand in AUTO_DETECT_CANDIDATES:
        if os.path.isdir(cand):
            DRIVE_CAMELS_PATH = cand
            print(f'Auto-detected camels_us at: {cand}')
            break
    else:
        raise RuntimeError(
            'Could not find camels_us. Set DRIVE_CAMELS_PATH explicitly.')
else:
    assert os.path.isdir(DRIVE_CAMELS_PATH)

topo_file = os.path.join(DRIVE_CAMELS_PATH, 'camels_attributes_v2.0', 'camels_topo.txt')
assert os.path.isfile(topo_file), f'Expected {topo_file} — does the folder have CAMELS contents?'
print(f'Verified camels_topo.txt present.')

DRIVE_RUNS = '/content/drive/MyDrive/neural_hydrology_runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)
print(f'Runs dir: {DRIVE_RUNS}')

SEEDS = [11] if MODE == 'demo' else [11, 13, 17]

# Locked condition definitions for the 5-cond factorial.
# (cond_id, variant_flag, run_subfolder)
GRAPH_CONDITIONS = [
    ('G',     'empty_graph',              'G'),
    ('G+T',   'topology_features',        'G_T'),
    ('G+M',   'warm',                     'G_M'),
    ('G+T+M', 'full_graph_with_topology', 'G_T_M'),
]
RUN_ROOT_REL = 'runs/5cond_factorial'  # paths are repo-relative

print(f'\nMODE = {MODE}; SEEDS = {SEEDS}')
print(f'Conditions: L (NH cudalstm) + ' + ', '.join(c[0] for c in GRAPH_CONDITIONS))
print(f'Total runs: {(1 + len(GRAPH_CONDITIONS)) * len(SEEDS)}')

Auto-detected camels_us at: /content/drive/MyDrive/camels_us
Verified camels_topo.txt present.
Runs dir: /content/drive/MyDrive/neural_hydrology_runs

MODE = full; SEEDS = [11, 13, 17]
Conditions: L (NH cudalstm) + G, G+T, G+M, G+T+M
Total runs: 15


## Cell 3 — Clone the repo from GitHub

In [3]:
REPO_DIR = '/content/nh'
import shutil
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone {GITHUB_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -n 3

Cloning into '/content/nh'...
remote: Enumerating objects: 4979, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 4979 (delta 25), reused 66 (delta 19), pack-reused 4879 (from 3)
Receiving objects: 100% (4979/4979), 595.72 MiB | 20.59 MiB/s, done.
Resolving deltas: 100% (445/445), done.
Updating files: 100% (4304/4304), done.
/content/nh
eff54e2 (HEAD -> main, origin/main, origin/HEAD) Cell 8 off by default; Cell 10 cascade-detection on fast-failures
7e58529 Fix Cell 10: use Component-0 L baseline (not 23-basin pilot) for graph cfg+scaler
702d65a pre-run env


## Cell 4 — Install dependencies (pin numpy<2 so torch C-bindings work)

In [4]:
%cd {REPO_DIR}
# Two-pass install to avoid the numpy/pandas ABI mismatch we hit in the prior run.
!pip install -q -e . pynhd networkx 2>&1 | tail -2
!pip install -q --force-reinstall --no-deps "numpy<2" "pandas==2.1.4" 2>&1 | tail -2

# Verify torch ↔ numpy ABI
import importlib, sys
for mod in list(sys.modules):
    if mod.startswith('numpy') or mod.startswith('pandas'):
        del sys.modules[mod]
import numpy as np, pandas as pd, torch
x = torch.from_numpy(np.array([1.0]))
_ = pd.DataFrame({'a': [1, 2]})
print(f'numpy {np.__version__}  pandas {pd.__version__}  torch {torch.__version__}  CUDA: {torch.cuda.is_available()}')

# torch.compile requires PyTorch >= 2.0; warn but continue if older.
torch_major = int(torch.__version__.split('.')[0])
USE_COMPILE = torch_major >= 2
print(f'torch.compile enabled: {USE_COMPILE}')

/content/nh
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.8/223.8 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 117.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 101.0 MB/s eta 0:00:00
numpy 1.26.4  pandas 2.1.4  torch 2.10.0+cu128  CUDA: True
torch.compile enabled: True


## Cell 5 — Symlink data and runs from Drive

In [5]:
%cd {REPO_DIR}
REPO_DATA = os.path.join(REPO_DIR, 'datasets', 'camels_us')
os.makedirs(os.path.dirname(REPO_DATA), exist_ok=True)
if os.path.islink(REPO_DATA) or os.path.isdir(REPO_DATA):
    !rm -rf {REPO_DATA}
os.symlink(DRIVE_CAMELS_PATH, REPO_DATA)

REPO_RUNS = os.path.join(REPO_DIR, 'runs')
if os.path.islink(REPO_RUNS) or os.path.isdir(REPO_RUNS):
    !rm -rf {REPO_RUNS}
os.symlink(DRIVE_RUNS, REPO_RUNS)

# Make sure the 5cond_factorial output folder exists under Drive.
os.makedirs(os.path.join(REPO_DIR, RUN_ROOT_REL), exist_ok=True)

print(f'datasets/camels_us -> {DRIVE_CAMELS_PATH}')
print(f'runs/ -> {DRIVE_RUNS}')
print(f'5cond outputs -> {os.path.join(DRIVE_RUNS, "5cond_factorial")}')
!ls datasets/camels_us | head -3

/content/nh
datasets/camels_us -> /content/drive/MyDrive/camels_us
runs/ -> /content/drive/MyDrive/neural_hydrology_runs
5cond outputs -> /content/drive/MyDrive/neural_hydrology_runs/5cond_factorial
basin_mean_forcing
camels_attributes_v2.0
usgs_streamflow


## Cell 6 — GPU check

In [6]:
!nvidia-smi -L
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime -> Change runtime type -> select a GPU (T4 recommended).')
print(f'GPU: {torch.cuda.get_device_name(0)}, '
       f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

GPU 0: Tesla T4 (UUID: GPU-259b4c08-e6fa-d2b0-d827-fed8b3289e7f)
GPU: Tesla T4, 15.6 GB


## Cell 7 — Generate per-seed YAML configs for Condition L

Templates land under `experiments/5cond_factorial/configs/L_seed{N}.yaml`. Each is a copy of `lstm_component0_baseline.yaml` with seed and experiment_name overridden, device pinned to `cuda:0`, and `run_dir` set to the locked output folder.

In [7]:
%cd {REPO_DIR}
CONFIG_DIR = os.path.join(REPO_DIR, 'experiments', '5cond_factorial', 'configs')
os.makedirs(CONFIG_DIR, exist_ok=True)
BASE_CONFIG = open('experiments/configs/lstm_component0_baseline.yaml').read()

for seed in SEEDS:
    cfg = BASE_CONFIG
    cfg = cfg.replace('experiment_name: lstm_component0_baseline',
                       f'experiment_name: L_seed{seed}')
    cfg = cfg.replace('run_dir: runs/', f'run_dir: {RUN_ROOT_REL}/')
    cfg = cfg.replace('device: cpu', 'device: cuda:0')
    # Add KGE alongside NSE so NH writes both into test_metrics.csv.
    cfg = cfg.replace('metrics:\n  - NSE\n', 'metrics:\n  - NSE\n  - KGE\n')
    if 'seed:' not in cfg:
        cfg += f'\nseed: {seed}\n'
    out = os.path.join(CONFIG_DIR, f'L_seed{seed}.yaml')
    with open(out, 'w') as f:
        f.write(cfg)
print(f'Wrote {len(SEEDS)} L-condition configs to {CONFIG_DIR}')

/content/nh
Wrote 3 L-condition configs to /content/nh/experiments/5cond_factorial/configs


## Cell 8 — Pre-flight smoke (DISABLED)

Smoke tests have been validated five times and the variants haven't changed.
Skipping saves ~10–12 minutes. The 23-basin pilot baseline is also no longer
needed at production time — Cell 10 uses the Component-0 `L_seed*` baselines
from Cell 9 for cfg + scaler.

If you ever change variant code in `train_graph_component0.py` and want to
re-validate, flip `RUN_SMOKE = True` below.

In [8]:
%cd {REPO_DIR}

# Smoke tests have been validated; default OFF for production sweeps.
RUN_SMOKE = False

if not RUN_SMOKE:
    print('Smoke tests SKIPPED (RUN_SMOKE=False). Production work starts at Cell 10.')
else:
    import glob, shutil, time

    # (1) 23-basin pilot baseline — required only for smoke tests.
    if not glob.glob(f'{REPO_DIR}/runs/05_lstm_23basin_strong_baseline'):
        print('Training 23-basin pilot baseline as a smoke-test prerequisite...')
        t0 = time.time()
        !python neuralhydrology/nh_run.py train --config-file experiments/configs/lstm_study_network_strong.yaml 2>&1 | tail -2
        pilot = sorted(glob.glob(f'{REPO_DIR}/runs/lstm_study_network_strong_*'))[-1]
        !mv {pilot} {REPO_DIR}/runs/05_lstm_23basin_strong_baseline
        print(f'    23-basin baseline done in {(time.time() - t0)/60:.1f} min')
    else:
        print('23-basin pilot baseline already exists — skipping.')

    # (2) Smoke-test each of the 4 graph variants (2 epochs each on the 23-basin pilot).
    compile_flag = '--use-compile' if USE_COMPILE else ''
    for cond_id, variant_flag, _sub in GRAPH_CONDITIONS:
        print(f'\nSmoke-testing variant `{variant_flag}` (Condition {cond_id})...')
        !python experiments/training/train_graph_component0.py \
            --variant {variant_flag} \
            --seed 42 \
            --smoke-test \
            --no-warm-start \
            {compile_flag} \
            --basin-file experiments/basin_lists/study_network_basins.txt \
            --edge-file topology_analysis/phase1_network_discovery/outputs/study_network_edges.csv \
            --baseline-run runs/05_lstm_23basin_strong_baseline 2>&1 | tail -3

    # (3) Immediate cleanup of every SMOKE folder.
    smoke_dirs = glob.glob(f'{REPO_DIR}/runs/graph_c0_*_seed42_SMOKE_*')
    for d in smoke_dirs:
        shutil.rmtree(d)
        print(f'  cleaned up: {os.path.basename(d)}')
    print(f'\nSmoke-test cleanup done. {len(smoke_dirs)} smoke folder(s) removed.')

/content/nh
Smoke tests SKIPPED (RUN_SMOKE=False). Production work starts at Cell 10.


## Cell 9 — Condition L: NH cudalstm baseline (no graph), train + evaluate × seeds

Condition L is the field-standard reference baseline. Outputs land at `runs/5cond_factorial/L_seed{N}/`.

In [9]:
%cd {REPO_DIR}
import time, glob

for seed in SEEDS:
    L_dir = f'{REPO_DIR}/{RUN_ROOT_REL}/L_seed{seed}'
    test_csv = f'{L_dir}/test/model_epoch030/test_metrics.csv'
    if os.path.isfile(test_csv):
        print(f'[skip] L seed={seed} already complete (train + evaluate)')
        continue

    cfg = f'{CONFIG_DIR}/L_seed{seed}.yaml'
    print(f'\n=== Condition L — seed={seed} (TRAIN) ===')
    t0 = time.time()

    # Train (only if not already trained)
    if not os.path.isfile(f'{L_dir}/model_epoch030.pt'):
        # NH writes to <run_dir>/<experiment_name>_<timestamp>/, then we move into place.
        !python neuralhydrology/nh_run.py train --config-file {cfg} 2>&1 | tail -2
        # NH placed a timestamped folder under runs/5cond_factorial/L_seed{seed}_*; rename to canonical path.
        timestamped = sorted(glob.glob(f'{REPO_DIR}/{RUN_ROOT_REL}/L_seed{seed}_*'))
        timestamped = [d for d in timestamped if d != L_dir and os.path.isdir(d)]
        if timestamped:
            if os.path.isdir(L_dir):
                shutil.rmtree(L_dir)
            os.rename(timestamped[-1], L_dir)
            print(f'    moved {os.path.basename(timestamped[-1])} -> L_seed{seed}/')
    else:
        print('    (already trained; running evaluate only)')

    # Evaluate to produce test_metrics.csv (required by the analysis cell)
    print(f'=== Condition L — seed={seed} (EVALUATE) ===')
    !python neuralhydrology/nh_run.py evaluate --run-dir {L_dir} --epoch 30 2>&1 | tail -3
    print(f'    {(time.time() - t0)/60:.1f} min total')

/content/nh
[skip] L seed=11 already complete (train + evaluate)
[skip] L seed=13 already complete (train + evaluate)
[skip] L seed=17 already complete (train + evaluate)


## Cell 10 — Conditions G, G+T, G+M, G+T+M: graph variants × seeds

All four DirectedGraphLSTM variants in one loop, each × 3 seeds = 12 runs.

- Each run writes directly to `runs/5cond_factorial/<sub>_seed{N}/` via `--run-dir` — no glob over timestamps.
- Skip-if-done is an exact path check on `<run-dir>/test_metrics.csv`.
- All graph runs use `--use-compile` (Phase 1.2 forward-pass speedup).
- Baseline reference for cfg+scaler is the **first completed `L_seed*` run from Cell 9** — that's a Component-0 cudalstm baseline whose `id_to_int` map knows all 183 basins. The 23-basin pilot baseline (used in Cell 8 smoke) only knows 23 basin IDs and would KeyError here.

In [10]:
%cd {REPO_DIR}
import glob, time, shutil

# Graph variants need a Component-0 (183-basin) baseline for cfg + scaler,
# NOT the 23-basin pilot baseline (whose id_to_int map only knows 23 basin
# IDs — it would KeyError on the first Component-0 basin).
# The L_seed* runs from Cell 9 ARE Component-0 cudalstm baselines, so any
# completed L_seed folder works as the cfg+scaler source. Pick the first
# canonical one found (cfg+scaler is identical across seeds; we use
# --no-warm-start so weights don't matter).
l_candidates = sorted(glob.glob(f'{REPO_DIR}/{RUN_ROOT_REL}/L_seed*'))
l_candidates = [d for d in l_candidates
                 if os.path.isdir(d) and (
                     glob.glob(f'{d}/model_epoch*.pt') or
                     glob.glob(f'{d}/train_data/*'))]
assert l_candidates, (
    'No completed L_seed* run found. Cell 9 must finish at least one '
    'Condition L run before Cell 10 (graph variants need its '
    'Component-0 cfg + scaler).')
BASELINE_FOR_GRAPH = l_candidates[0]
print(f'Baseline for graph variants (cfg + scaler): {BASELINE_FOR_GRAPH}')

compile_flag = '--use-compile' if USE_COMPILE else ''

# Cascade detection: if two consecutive runs finish in <60s without producing
# test_metrics.csv, the IPython kernel is almost certainly wedged (the same
# failure mode that hit us overnight — a single runtime hiccup propagates
# SIGINT to every subsequent !python subprocess). Abort loudly so we don\'t
# silently lose all 11 remaining runs.
consecutive_fast_fails = 0
FAST_FAIL_SECONDS = 60

for cond_id, variant_flag, sub in GRAPH_CONDITIONS:
    for seed in SEEDS:
        run_dir = f'{REPO_DIR}/{RUN_ROOT_REL}/{sub}_seed{seed}'
        test_csv = f'{run_dir}/test_metrics.csv'
        if os.path.isfile(test_csv):
            print(f'[skip] {cond_id} seed={seed} already complete: {sub}_seed{seed}/test_metrics.csv')
            continue
        # Wipe any stale/empty dir from a previously-failed launch so the
        # trainer starts fresh (the success case is already handled above).
        if os.path.isdir(run_dir):
            shutil.rmtree(run_dir)
        print(f'\n=== Condition {cond_id} (variant={variant_flag}) — seed={seed} ===')
        t0 = time.time()
        !python experiments/training/train_graph_component0.py \
            --variant {variant_flag} \
            --seed {seed} --no-warm-start --epochs 30 \
            --run-dir {run_dir} \
            {compile_flag} \
            --baseline-run {BASELINE_FOR_GRAPH} 2>&1 | tail -3
        elapsed = time.time() - t0
        success = os.path.isfile(test_csv)
        if success:
            consecutive_fast_fails = 0
            print(f'    OK  {elapsed/60:.1f} min')
        elif elapsed < FAST_FAIL_SECONDS:
            consecutive_fast_fails += 1
            print(f'    FAST-FAIL  {elapsed:.0f}s with no test_metrics.csv '
                   f'(consecutive: {consecutive_fast_fails})')
            if consecutive_fast_fails >= 2:
                raise RuntimeError(
                    f'\n*** ABORTING Cell 10: {consecutive_fast_fails} consecutive fast-failures.\n'
                    f'*** Almost certainly an IPython kernel cascade after a Colab runtime hiccup.\n'
                    f'*** Action: Runtime → Restart runtime, then re-run cells 1–7 and Cell 10.\n'
                    f'*** Cell 10 is idempotent — already-finished runs (with test_metrics.csv) will skip.\n')
        else:
            # Long run that failed normally (OOM, NaN loss, etc.) — log but continue.
            consecutive_fast_fails = 0
            print(f'    WARN  {elapsed/60:.1f} min elapsed but no test_metrics.csv. '
                   f'Investigate this run, continuing to next.')

/content/nh
Baseline for graph variants (cfg + scaler): /content/nh/runs/5cond_factorial/L_seed11
[skip] G seed=11 already complete: G_seed11/test_metrics.csv
[skip] G seed=13 already complete: G_seed13/test_metrics.csv
[skip] G seed=17 already complete: G_seed17/test_metrics.csv
[skip] G+T seed=11 already complete: G_T_seed11/test_metrics.csv
[skip] G+T seed=13 already complete: G_T_seed13/test_metrics.csv
[skip] G+T seed=17 already complete: G_T_seed17/test_metrics.csv
[skip] G+M seed=11 already complete: G_M_seed11/test_metrics.csv

=== Condition G+M (variant=warm) — seed=13 ===
2026-05-18 10:32:49,735: DONE. variant=warm  seed=13  median_nse=0.611
2026-05-18 10:32:49,735: Mean epoch wall-clock: 260.4s  (4.3 min)
2026-05-18 10:32:49,736: ======================================================================
    OK  137.9 min

=== Condition G+M (variant=warm) — seed=17 ===
2026-05-18 12:44:18,476: DONE. variant=warm  seed=17  median_nse=0.581
2026-05-18 12:44:18,476: Mean epoch wall-

## Cell 11 — Aggregate results (cross-condition analysis)

Calls `experiments/analysis/compare_5conditions.py`, which loads all 15 runs from `runs/5cond_factorial/`, computes NSE / KGE / log-NSE with bootstrap CIs, the six pairwise contrasts, the interaction term, and depth/area-stratified plots, then writes `experiments/analysis_outputs/5cond_component0/RESULTS.md` plus figures.

In [11]:
%cd {REPO_DIR}
ANALYSIS_SCRIPT = f'{REPO_DIR}/experiments/analysis/compare_5conditions.py'
if os.path.isfile(ANALYSIS_SCRIPT):
    !python {ANALYSIS_SCRIPT} 2>&1 | tail -40
else:
    print(f'NOTE: {ANALYSIS_SCRIPT} not found yet (Phase 3 still pending).')
    print('Falling back to a quick inline summary of per-run median NSE.')

/content/nh
Run discovery from /content/nh/runs/5cond_factorial:
       L: 3 seed(s) found: [11, 13, 17]
       G: 3 seed(s) found: [11, 13, 17]
     G+T: 3 seed(s) found: [11, 13, 17]
     G+M: 3 seed(s) found: [11, 13, 17]
   G+T+M: 3 seed(s) found: [11, 13, 17]


=== HEADLINE ===
       L  NSE +0.653  KGE +0.728  logNSE +0.683
       G  NSE +0.609  KGE +0.515  logNSE +0.622
     G+T  NSE +0.605  KGE +0.512  logNSE +0.616
     G+M  NSE +0.583  KGE +0.559  logNSE +0.605
   G+T+M  NSE +0.586  KGE +0.533  logNSE +0.604

Outputs at: /content/nh/experiments/analysis_outputs/5cond_component0


## Cell 12 — Inline quick-look summary (always runs, even if Phase 3 script not present)

In [12]:
%cd {REPO_DIR}
import json, glob, os
import pandas as pd, numpy as np
from pathlib import Path

def load_metrics_at(path):
    if not os.path.isfile(path):
        return None
    df = pd.read_csv(path, dtype={'basin': str})
    if 'NSE' not in df.columns:
        return None
    return df.set_index('basin')['NSE'].to_dict()

results = {}
# Condition L: NH writes to <run-dir>/test/model_epoch030/test_metrics.csv
for seed in SEEDS:
    nse = load_metrics_at(f'{REPO_DIR}/{RUN_ROOT_REL}/L_seed{seed}/test/model_epoch030/test_metrics.csv')
    if nse is not None:
        results.setdefault('L', {})[seed] = nse
# Graph conditions: train_graph_component0.py writes to <run-dir>/test_metrics.csv
for cond_id, _variant, sub in GRAPH_CONDITIONS:
    for seed in SEEDS:
        nse = load_metrics_at(f'{REPO_DIR}/{RUN_ROOT_REL}/{sub}_seed{seed}/test_metrics.csv')
        if nse is not None:
            results.setdefault(cond_id, {})[seed] = nse

for cond_id, r in results.items():
    if r:
        print(f'{cond_id:>6}: {len(r)} seed(s) found: {sorted(r.keys())}')
    else:
        print(f'{cond_id:>6}: NO RESULTS FOUND')

summary = {}
for cond_id, r in results.items():
    if not r:
        continue
    medians = sorted(np.median(list(d.values())) for d in r.values())
    summary[cond_id] = {
        'n_seeds': len(medians),
        'seeds': sorted(r.keys()),
        'median_NSE_per_seed': [float(x) for x in medians],
        'cross_seed_median': float(np.median(medians)),
        'cross_seed_std': float(np.std(medians)) if len(medians) > 1 else 0.0,
    }
print('\n' + json.dumps(summary, indent=2))

OUT_DIR = Path(REPO_DIR) / 'experiments' / 'analysis_outputs' / '5cond_component0'
OUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUT_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
rows = [{'condition': cond_id, 'seed': s, 'basin': b, 'NSE': v}
        for cond_id, r in results.items() for s, d in r.items() for b, v in d.items()]
if rows:
    pd.DataFrame(rows).to_csv(OUT_DIR / 'per_basin_per_seed.csv', index=False)
    print(f'\nWrote {len(rows)} rows to per_basin_per_seed.csv')

# Headline contrasts (cross-seed median NSE)
needed = {'L', 'G', 'G+T', 'G+M', 'G+T+M'}
if needed.issubset(summary):
    L = summary['L']['cross_seed_median']
    G = summary['G']['cross_seed_median']
    GT = summary['G+T']['cross_seed_median']
    GM = summary['G+M']['cross_seed_median']
    GTM = summary['G+T+M']['cross_seed_median']
    print(f'\n*** Cross-seed median NSE per condition ***')
    print(f'    L     : {L:+.3f}    (NH cudalstm reference baseline)')
    print(f'    G     : {G:+.3f}    (architecture-matched no-graph control)')
    print(f'    G+T   : {GT:+.3f}')
    print(f'    G+M   : {GM:+.3f}')
    print(f'    G+T+M : {GTM:+.3f}')
    print(f'\n*** Pairwise contrasts ***')
    print(f'    L − G               : {L - G:+.3f}    (architecture/methodology delta)')
    print(f'    (G+T) − G           : {GT - G:+.3f}    (effect of topology features alone)')
    print(f'    (G+M) − G           : {GM - G:+.3f}    (effect of message passing alone)')
    print(f'    (G+T+M) − (G+T)     : {GTM - GT:+.3f}    (adding messages on top of T)')
    print(f'    (G+T+M) − (G+M)     : {GTM - GM:+.3f}    (adding T on top of messages)')
    print(f'    (G+T+M) − G         : {GTM - G:+.3f}    (combined effect)')
    interaction = GTM - GT - GM + G
    print(f'    interaction (T×M)    : {interaction:+.3f}    (super-/sub-additivity of T and M)')
else:
    missing = needed - set(summary.keys())
    print(f'\nMissing conditions: {missing}. Re-run those cells.')

/content/nh
     L: 3 seed(s) found: [11, 13, 17]
     G: 3 seed(s) found: [11, 13, 17]
   G+T: 3 seed(s) found: [11, 13, 17]
   G+M: 3 seed(s) found: [11, 13, 17]
 G+T+M: 3 seed(s) found: [11, 13, 17]

{
  "L": {
    "n_seeds": 3,
    "seeds": [
      11,
      13,
      17
    ],
    "median_NSE_per_seed": [
      0.6511677447636253,
      0.6529475952853108,
      0.6555066057116874
    ],
    "cross_seed_median": 0.6529475952853108,
    "cross_seed_std": 0.0017808274159460894
  },
  "G": {
    "n_seeds": 3,
    "seeds": [
      11,
      13,
      17
    ],
    "median_NSE_per_seed": [
      0.5897120863231458,
      0.608603960427075,
      0.6141429060121809
    ],
    "cross_seed_median": 0.608603960427075,
    "cross_seed_std": 0.010458636920064488
  },
  "G+T": {
    "n_seeds": 3,
    "seeds": [
      11,
      13,
      17
    ],
    "median_NSE_per_seed": [
      0.6032093813277466,
      0.6051525881730949,
      0.6126082154187782
    ],
    "cross_seed_median": 0.60515258

## Done

Result files are at:
- `/content/drive/MyDrive/neural_hydrology_runs/experiments/analysis_outputs/5cond_component0/summary.json`
- `/content/drive/MyDrive/neural_hydrology_runs/experiments/analysis_outputs/5cond_component0/per_basin_per_seed.csv`
- `/content/drive/MyDrive/neural_hydrology_runs/experiments/analysis_outputs/5cond_component0/RESULTS.md` (if Phase 3 script ran)
- All 15 run folders under `/content/drive/MyDrive/neural_hydrology_runs/5cond_factorial/`

Pull these to your local repo (drag from Drive desktop sync, or copy via Drive web UI). Then in chat type **`crs interpret 5cond results`** and the chief-research-scientist will read them.